# Análisis Bivariante: Visualización de Asociaciones para Plomo (Pb)

Visualizaciones bivariantes epidemiológicas: contrastes no paramétricos y paramétricos, gradientes dietarios (Consumo de Carnes / Alimentos), co-exposición y mosaicos multipanel.

> Protocolo bioestadístico y epidemiológico de biomonitoreo con **HeavyStats** v0.4.3.

# Análisis Bivariante: Visualización Epidemiológica y Toxicológica de Asociaciones (Plomo - Pb)
### Paquete Bioestadístico HeavyStats (`heavystats.bivariate`)

Motor gráfico editorial a 300 DPI estructurado según los principios del análisis bivariante bioestadístico:
1. **Variable Cuantitativa vs. Variable Cualitativa**:
   - Contrastes no paramétricos (Mann–Whitney $U$, Kruskal–Wallis $H$, diferencias de medianas $\Delta\text{Med}$ y tamaños de efecto $r_{rb}$ / $\epsilon^2$).
   - Contrastes paramétricos (prueba $t$ de Welch, ANOVA unidireccional, diferencia de medias y Hedges' $g$ / $\eta^2$).
   - Anotación directa en corchetes (*brackets*), tamaños muestrales ($n=\dots$) en ejes y estilización de publicaciones científicas.
   - Gradientes ordinales dietarios (prueba de Jonckheere–Terpstra), mosaicos de consumo y perfiles polares (radar / spider charts).
2. **Variable Cuantitativa vs. Variable Cuantitativa**:
   - Dispersión continua, correlación de Spearman $\rho_s$ con intervalos de confianza al 95% calculados por **Bootstrap (2,000 réplicas)**, Kendall $\tau_b$ y regresión lineal OLS / Pearson $r$ ($R^2$).
   - Matrices de co-exposición bivariante: Heatmap triangular inferior estilizado y matriz de dispersión compacta con distribuciones KDE univariantes.
3. **Mosaico Bivariante Integrado y Funciones a Nivel de Módulo**:
   - Cuadrícula multipanel consolidada (`plot_grid`) y funciones funcionales de acceso directo (`compare_groups_plot`, `correlation_analysis_plot`, `diet_radar_plot`, etc.).


---
## 1. Carga de Datos y Configuración del Motor Gráfico Bivariante (Plomo)

Inicializamos `BivariatePlots` para renderizar figuras analíticas coordinadas a 300 DPI con estética editorial predefinida.


In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Asegurar carga del módulo de desarrollo de HeavyStats
heavystats_src = Path('d:/vscode/results/HeavyStats/src')
if heavystats_src.exists() and str(heavystats_src) not in sys.path:
    sys.path.insert(0, str(heavystats_src))

import heavystats as hs
from heavystats.bivariate import (
    BivariatePlots,
    compare_groups_plot,
    correlation_analysis_plot,
    coexposure_matrix_plot,
    diet_radar_plot,
    diet_boxplots_plot,
)

# Definición de rutas base y directorios de salida
base_dir = Path.cwd() if (Path.cwd() / "datos").exists() else Path.cwd().parent
ruta_datos = base_dir / "datos" / "procesados" / "datos_plomo_n20.csv"

if not ruta_datos.exists():
    df_raw = hs.load_data(base_dir / "datos" / "datos_plomo.csv")
    df_proc = hs.create_composite_indicators(
        hs.encode_dietary_frequencies(
            hs.desaggregate_multiple_responses(
                hs.standardize_boolean_columns(df_raw)
            )
        )
    )
    df_analytical = hs.select_metal(df_proc, concentration_col="plomo")
else:
    df_analytical = pd.read_csv(ruta_datos, encoding="utf-8")

# Instanciación del motor bivariante con paleta editorial predefinida
bp = BivariatePlots(
    df=df_analytical,
    palette="crest",
    style="ticks",
    context="notebook"
)

salidas_graficos = base_dir / "salidas" / "bivariante" / "graficos"
salidas_graficos.mkdir(parents=True, exist_ok=True)

print(f"Motor BivariatePlots inicializado exitosamente (N={len(df_analytical)} observaciones, {len(df_analytical.columns)} variables).")
print(f"Directorio de figuras configurado en: {salidas_graficos}")


---
## 2. Variable Cuantitativa vs. Variable Cualitativa (Carga Corporal)

Evaluación de la concentración sérica del biomarcador frente a factores sociodemográficos, clínicos y ambientales.
- **Factores Dicotómicos (`plot_binary`)**: Diagrama de cajas con puntos individuales (*stripplot*), medianas [RIQ], prueba de Mann–Whitney $U$, correlación biserial de rangos ($r_{rb}$) y diferencia de medianas. Diseñado estrictamente para contraste de 2 grupos (sin línea de límite toxicológico).
- **Factores Politómicos (`plot_categorical`)**: Comparación entre >2 grupos con prueba de Kruskal–Wallis $H$ y tamaño del efecto $\epsilon^2$.
- **Contraste Paramétrico (`compare_groups(method="parametric")`)**: Medias $\pm$ DE con prueba $t$ de Welch o ANOVA unidireccional y Hedges' $g$ / $\eta^2$.


In [ ]:
# 1.1 Factores Dicotómicos: Concentración de Plomo según Sexo (Femenino vs Masculino)
fig_sexo, ax_sexo = bp.plot_binary(
    metal="Plomo_ug_dL",
    group_col="Sexo",
    show_points=True,
    title="Concentración Sérica de Plomo según Sexo",
    filepath=str(salidas_graficos / "boxplot_plomo_sexo.png")
)
fig_sexo.savefig(salidas_graficos / "boxplot_plomo_sexo.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 1.2 Factores Dicotómicos: Plomo según Condición de Exposición Ocupacional/Ambiental
fig_exp, ax_exp = bp.plot_binary(
    metal="Plomo_ug_dL",
    group_col="Es_Expuesto",
    show_points=True,
    title="Niveles de Plomo según Condición de Expuesto",
    filepath=str(salidas_graficos / "boxplot_plomo_es_expuesto.png")
)
fig_exp.savefig(salidas_graficos / "boxplot_plomo_es_expuesto.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 1.3 Factores Politómicos: Concentración de Plomo según Sector Geográfico (>2 grupos)
fig_sec, ax_sec = bp.plot_categorical(
    metal="Plomo_ug_dL",
    group_col="Sector",
    show_points=True,
    title="Distribución de Plomo por Sector de Residencia (Kruskal–Wallis)",
    filepath=str(salidas_graficos / "boxplot_plomo_sector.png")
)
fig_sec.savefig(salidas_graficos / "boxplot_plomo_sector.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 1.4 Contraste Paramétrico: Prueba t de Welch con Media ± DE y Tamaño de Efecto Hedges' g
fig_param, ax_param = bp.compare_groups(
    quantitative="Plomo_ug_dL",
    group="Sexo",
    method="parametric",
    show_points=True,
    title="Contraste Paramétrico: Plomo por Sexo (Welch t / Hedges' g)",
    filepath=str(salidas_graficos / "boxplot_plomo_sexo_parametrico.png")
)
fig_param.savefig(salidas_graficos / "boxplot_plomo_sexo_parametrico.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 1.5 Factores de Exposición por Variables Dummy de Proximidad Ambiental
fig_dummies, ax_dummies = bp.plot_dummies(
    metal="Plomo_ug_dL",
    dimension="all",
    show_points=True,
    title="Carga de Plomo según Fuentes Ambientales de Exposición",
    filepath=str(salidas_graficos / "boxplot_plomo_factores_ambientales.png")
)
fig_dummies.savefig(salidas_graficos / "boxplot_plomo_factores_ambientales.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 1.6 Interacción de Factores: Estratificación Cruzada [Sexo - Condición de Expuesto]
fig_inter, ax_inter = bp.compare_groups(
    quantitative="Plomo_ug_dL",
    group=["Sexo", "Es_Expuesto"],
    show_points=True,
    title="Interacción Sexo × Exposición frente a Plomo en Sangre",
    filepath=str(salidas_graficos / "boxplot_plomo_interaccion_sexo_expuesto.png")
)
fig_inter.savefig(salidas_graficos / "boxplot_plomo_interaccion_sexo_expuesto.pdf", dpi=300, bbox_inches="tight")
plt.show()


---
## 3. Hábitos Dietarios y Validación del Algoritmo de Riesgo

- **Gradiente Dosis–Respuesta Monótono**: Prueba de Jonckheere–Terpstra ($JT$) para contrastar tendencias crecientes/decrecientes ordenadas por frecuencia de consumo.
- **Mosaicos Dietarios ("Todo en Uno")**: Formato facetado en cuadrícula (`layout="grid"`) o consolidado horizontal (`layout="consolidated"`), adoptando la paleta editorial predefinida.
- **Perfil Multivariante Polar (Radar / Spider Chart)**: Comparación de patrones alimentarios mediante `ax.set_thetagrids()` estratificada por mediana o terciles de concentración.
- **Validación del Score de Riesgo**: Regresión empírica frente al biomarcador biológico con bandas de confianza al 95%.


In [ ]:
# 2.1 Tendencia Monótona Ordinal: Frecuencia de Consumo de Consumo de Carnes / Alimentos vs Plomo
fig_dieta, ax_dieta = bp.plot_dietary(
    ordinal_col="Alim_Carnes",
    metal="Plomo_ug_dL",
    show_points=True,
    title="Gradiente Ordinal de Ingesta de Consumo de Carnes / Alimentos vs Plomo Sérico (Jonckheere–Terpstra)",
    filepath=str(salidas_graficos / "tendencia_ordinal_plomo_Alim_Carnes.png")
)
fig_dieta.savefig(salidas_graficos / "tendencia_ordinal_plomo_Alim_Carnes.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 2.2 Mosaico Dietario Multivariable Facetado: Todos los Grupos de Alimentos vs Plomo
fig_diet_grid, axes_diet_grid = bp.plot_diet_boxplots(
    metal="Plomo_ug_dL",
    layout="grid",
    ncols=4,
    filepath=str(salidas_graficos / "mosaico_dietario_grid_plomo.png")
)
fig_diet_grid.savefig(salidas_graficos / "mosaico_dietario_grid_plomo.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 2.3 Diagrama Dietario Consolidado Horizontal (Frecuente/Diario vs Ocasional/Nunca)
fig_diet_cons, ax_diet_cons = bp.plot_diet_boxplots(
    metal="Plomo_ug_dL",
    layout="consolidated",
    filepath=str(salidas_graficos / "diagrama_dietario_consolidado_plomo.png")
)
fig_diet_cons.savefig(salidas_graficos / "diagrama_dietario_consolidado_plomo.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 2.4 Perfil Dietario en Coordenadas Polares (Radar Chart) Estratificado por Mediana
fig_radar, axes_radar = bp.plot_diet_radar(
    metal="Plomo_ug_dL",
    stratify_by="median",
    split_panels=True,
    figsize=(15.0, 7.5),
    filepath=str(salidas_graficos / "radar_dietario_plomo_split.png")
)
fig_radar.savefig(salidas_graficos / "radar_dietario_plomo_split.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 2.5 Validación Empírica del Algoritmo de Riesgo frente a Concentraciones Biológicas
fig_risk, axes_risk = bp.plot_risk_algorithm(
    metals=["Plomo_ug_dL"],
    figsize=(6.5, 4.5),
    filepath=str(salidas_graficos / "validacion_algoritmo_riesgo_plomo.png")
)
fig_risk.savefig(salidas_graficos / "validacion_algoritmo_riesgo_plomo.pdf", dpi=300, bbox_inches="tight")
plt.show()


---
## 4. Variable Cuantitativa vs. Variable Cuantitativa (Gradientes Continuos y Co-Exposición)

- **Correlación Continua Bivariante**: Ajuste de regresión lineal con banda de confianza al 95%, coeficiente de Spearman $\rho_s$ con IC 95% Bootstrap (2,000 remuestreos) y Kendall $\tau_b$.
- **Matriz de Co-Exposición Bivariante**:
  - `kind="heatmap"`: Matriz triangular inferior con paleta divergente, coeficientes e intervalos Bootstrap.
  - `kind="scatter"`: Matriz compacta de dispersión bivariante con densidades KDE univariantes en la diagonal y resúmenes estadísticos en el triángulo superior.


In [ ]:
# 3.1 Dispersión y Correlación No Paramétrica: Edad vs Concentración de Plomo
fig_corr_sp, ax_corr_sp = bp.correlation_analysis(
    x="Edad",
    y="Plomo_ug_dL",
    method="nonparametric",
    n_boot=2000,
    title="Correlación No Paramétrica: Edad vs Plomo (Bootstrap IC 95%)",
    filepath=str(salidas_graficos / "dispersion_plomo_edad_spearman.png")
)
fig_corr_sp.savefig(salidas_graficos / "dispersion_plomo_edad_spearman.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 3.2 Regresión Paramétrica: Score de Riesgo vs Plomo (Pearson r, Ecuación OLS y R²)
fig_corr_pr, ax_corr_pr = bp.correlation_analysis(
    x="Score_Riesgo",
    y="Plomo_ug_dL",
    method="parametric",
    title="Regresión Lineal OLS: Score de Riesgo vs Plomo Sérico",
    filepath=str(salidas_graficos / "regresion_plomo_score_riesgo_pearson.png")
)
fig_corr_pr.savefig(salidas_graficos / "regresion_plomo_score_riesgo_pearson.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 3.3 Matriz de Co-Exposición Cuantitativa: Heatmap Triangular Editorial
cols_cuant = [c for c in ["Plomo_ug_dL", "Edad", "Score_Riesgo", "IMC", "Peso_kg", "Altura_cm"] if c in df_analytical.columns]

fig_coexp_heat, ax_coexp_heat = bp.plot_coexposure_matrix(
    variables=cols_cuant,
    kind="heatmap",
    method="nonparametric",
    n_boot=2000,
    title="Matriz de Co-Exposición Bivariante (Spearman rho_s con IC 95% Bootstrap)",
    filepath=str(salidas_graficos / "matriz_coexposicion_plomo_heatmap.png")
)
fig_coexp_heat.savefig(salidas_graficos / "matriz_coexposicion_plomo_heatmap.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 3.4 Matriz de Dispersión y Co-Exposición Compacta (Scatter + KDE + Estadísticos)
core_cuant = [c for c in ["Plomo_ug_dL", "Edad", "Score_Riesgo", "IMC"] if c in df_analytical.columns]

fig_coexp_scat, axes_coexp_scat = bp.plot_coexposure_matrix(
    variables=core_cuant,
    kind="scatter",
    method="nonparametric",
    n_boot=2000,
    title="Matriz Bivariante Compacta: Dispersión, Densidad y Correlación",
    filepath=str(salidas_graficos / "matriz_coexposicion_plomo_dispersion_compacta.png")
)
fig_coexp_scat.savefig(salidas_graficos / "matriz_coexposicion_plomo_dispersion_compacta.pdf", dpi=300, bbox_inches="tight")
plt.show()


---
## 5. Mosaico Bivariante Integrado y Funciones a Nivel de Módulo

- **Cuadrícula Exploratoria Multideterminante (`plot_grid`)**: Agrupa en un único lienzo factores binarios, politómicos, continuos y ordinales.
- **Funciones de Conveniencia a Nivel de Módulo**: Acceso funcional directo sin necesidad de instanciar `BivariatePlots` (ideal para scripts analíticos y pipelines automatizados).


In [ ]:
# 4.1 Mosaico Bivariante Integrado: Factores Sociodemográficos, Clínicos y Ambientales
fig_grid, axes_grid = bp.plot_grid(
    metal="Plomo_ug_dL",
    binary_cols=["Sexo", "Es_Expuesto"],
    categorical_cols=["Institucion", "Sector"],
    continuous_cols=["Edad", "Score_Riesgo"],
    ordinal_cols=["Alim_Carnes"],
    ncols=2,
    filepath=str(salidas_graficos / "mosaico_editorial_bivariante_plomo.png")
)
fig_grid.savefig(salidas_graficos / "mosaico_editorial_bivariante_plomo.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 4.2 Uso de Funciones Funcionales Directas del Módulo (API Funcional sin Instanciación)
fig_fn_comp, ax_fn_comp = compare_groups_plot(
    df=df_analytical,
    quantitative="Plomo_ug_dL",
    group="Es_Expuesto",
    title="Llamada Funcional Directa: compare_groups_plot()"
)
plt.show()

fig_fn_corr, ax_fn_corr = correlation_analysis_plot(
    df=df_analytical,
    x="Score_Riesgo",
    y="Plomo_ug_dL",
    title="Llamada Funcional Directa: correlation_analysis_plot()"
)
plt.show()

print(f"\n¡Figuras bivariantes de Plomo guardadas exitosamente en:\n  -> {salidas_graficos.resolve()}")
